# V12: Spectral Sparsity on the Cotangent Bundle — Tokens as Fourier Wells

## Core Principle

Sparsity belongs in **Fourier space**, not spatial space. A token is a sparse
collection of excited frequency modes whose superposition (IFFT) generates the
full spatial field. This shift resolves the Heisenberg contradiction in v11
(spatial sparsity → spectral spread → inefficient transport) and makes
activation, routing, and training structurally local.

## Forward Pass

```
token → sparse SPECTRAL section (s_k modes per subbundle) →
  [ context accumulator (SSM on spatial projection) →
    spectral transport (direct multiply on active modes, O(s)) →
    field reconstruction (IFFT → dense spatial cloud) →
    memory routing (spectral atoms, routed by context) →
    Langevin settling (spatial domain, spectral proximal every step) →
    FFT + re-sparsify → sparse spectral output ] × N →
  IFFT → final_norm → decode
```

## Key Differences from V11

| V11 | V12 |
|---|---|
| Sparse in spatial fiber (top-k dims) | Sparse in spectral fiber (top-s_k frequencies) |
| Diffusion = temporal conv + fiber FFT | Field reconstruction = IFFT from sparse spectral |
| Proximal = spatial soft-threshold | Proximal = FFT → top-s_k → IFFT (spectral) |
| Wilson line = causal conv | Context accumulator = SSM (parallel scan) |
| Transport touches all freq bins | Transport touches only active modes: O(s) |
| ~11.8M params | ~60K params (V12-Nano) |

In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
from dataclasses import dataclass
import math
import os
import urllib.request
import time
from tqdm.auto import tqdm

torch.manual_seed(42)
device = torch.device("mps" if torch.backends.mps.is_available() else
                      "cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

# ── Load Tiny Shakespeare ──
data_dir = os.path.join(os.path.dirname(os.path.abspath("__file__")), "..", "v8_real_text", "data")
data_path = os.path.join(data_dir, "input.txt")

if not os.path.exists(data_path):
    os.makedirs(data_dir, exist_ok=True)
    url = "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt"
    print("Downloading Tiny Shakespeare...")
    urllib.request.urlretrieve(url, data_path)

with open(data_path, "r") as f:
    text = f.read()

chars = sorted(set(text))
vocab_size = len(chars)
stoi = {c: i for i, c in enumerate(chars)}
itos = {i: c for c, i in stoi.items()}
def encode(s): return [stoi[c] for c in s]
def decode(t): return "".join(itos[i] for i in t if i in itos)

data = torch.tensor(encode(text), dtype=torch.long)
split = int(0.9 * len(data))
train_data, val_data = data[:split], data[split:]
print(f"Train: {len(train_data):,} | Val: {len(val_data):,} chars")
print(f"Vocab: {vocab_size} unique characters")

Device: mps
Train: 1,003,854 | Val: 111,540 chars
Vocab: 65 unique characters


In [ ]:
@dataclass
class V12Config:
    # Fiber geometry
    fiber_dim: int = 256
    n_subbundles: int = 8
    spectral_sparsity: int = 8      # s_k: active spectral modes per subbundle

    # Vocabulary
    vocab_size: int = 65
    max_seq_len: int = 128

    # Architecture
    n_blocks: int = 4
    context_dim: int = 128
    atoms_per_subbundle: int = 32
    k_active_atoms: int = 8

    # Langevin settling
    langevin_steps: int = 5
    langevin_lr: float = 0.03
    beta_init: float = 1.0
    beta_final: float = 12.0

    # Training
    learning_rate: float = 3e-3
    warmup_steps: int = 500
    dropout: float = 0.1
    batch_size: int = 32
    seq_len: int = 128
    max_steps: int = 5000
    eval_interval: int = 250
    eval_steps: int = 10

    @property
    def subbundle_dim(self):
        return self.fiber_dim // self.n_subbundles

    @property
    def total_active_modes(self):
        return self.n_subbundles * self.spectral_sparsity


config = V12Config(vocab_size=vocab_size)
print(f"Fiber: {config.fiber_dim} = {config.n_subbundles} × {config.subbundle_dim}")
print(f"Spectral sparsity: {config.spectral_sparsity}/{config.subbundle_dim} modes per subbundle "
      f"= {100*config.spectral_sparsity/config.subbundle_dim:.0f}% active "
      f"({config.total_active_modes}/{config.fiber_dim} total)")
print(f"Atoms: {config.atoms_per_subbundle} per subbundle, k={config.k_active_atoms} active")
print(f"Blocks: {config.n_blocks}, Langevin steps: {config.langevin_steps}")
print(f"Batch: {config.batch_size}, Seq: {config.seq_len}")

def get_batch(split_data, cfg):
    max_start = len(split_data) - cfg.seq_len - 1
    starts = torch.randint(0, max_start, (cfg.batch_size,))
    return torch.stack([split_data[s:s+cfg.seq_len] for s in starts]).to(device)

Fiber: 256 = 8 × 32
Spectral sparsity: 8/32 modes per subbundle = 25% active (64/256 total)
Atoms: 32 per subbundle, k=8 active
Blocks: 4, Langevin steps: 5
Batch: 32, Seq: 128


## V12 Architecture Modules

The key innovation: sparsity is enforced in the **spectral** (Fourier) domain.
Tokens are embedded as complex spectral vectors, transported via direct multiply
on active modes, reconstructed to dense spatial via IFFT, settled via Langevin
in spatial domain, then projected back to spectral sparsity at every step.

In [3]:
# ── Shared utilities ─────────────────────────────────────────────────

def spectral_sparsify(x_complex, cfg):
    """Per-subbundle top-s_k sparsification in spectral domain.
    x_complex: (..., fiber_dim) complex tensor.
    Returns sparsified complex tensor with exactly s_k nonzero modes per subbundle.
    
    Uses pairwise comparison instead of torch.topk — 150× faster on MPS.
    Mask computed under no_grad (it's a non-differentiable selection;
    gradients flow through the selected values only)."""
    shape = x_complex.shape
    x_subs = x_complex.reshape(*shape[:-1], cfg.n_subbundles, cfg.subbundle_dim)
    with torch.no_grad():
        mags = x_subs.abs()
        # Count how many elements are strictly greater than each element
        # An element is in the top-s_k if fewer than s_k elements exceed it
        gt_count = (mags.unsqueeze(-1) < mags.unsqueeze(-2)).sum(dim=-1)
        mask = (gt_count < cfg.spectral_sparsity).float()
    return (x_subs * mask).reshape(shape)


def spectral_to_spatial(x_spectral, cfg):
    """IFFT per subbundle: sparse spectral → dense spatial (real)."""
    shape = x_spectral.shape
    subs = x_spectral.reshape(*shape[:-1], cfg.n_subbundles, cfg.subbundle_dim)
    spatial = torch.fft.ifft(subs, dim=-1).real
    return spatial.reshape(*shape[:-1], cfg.fiber_dim)


def spatial_to_spectral(x_spatial, cfg):
    """FFT per subbundle: dense spatial → spectral (complex)."""
    shape = x_spatial.shape
    subs = x_spatial.reshape(*shape[:-1], cfg.n_subbundles, cfg.subbundle_dim)
    spectral = torch.fft.fft(subs, dim=-1)
    return spectral.reshape(*shape[:-1], cfg.fiber_dim)


def spectral_proximal(x_spatial, cfg):
    """The V12 proximal operator: FFT → top-s_k per subbundle → IFFT.
    Enforces spectral sparsity while operating on spatial state.
    Gradient flows through selected modes only (structurally sparse backprop)."""
    x_spec = spatial_to_spectral(x_spatial, cfg)
    x_sparse = spectral_sparsify(x_spec, cfg)
    return spectral_to_spatial(x_sparse, cfg)


def parallel_associative_scan(A, B):
    """O(log T) Hillis-Steele parallel scan: q_t = A_t * q_{t-1} + B_t.
    Uses doubling — log2(T) fully vectorized steps instead of T sequential iterations."""
    _, T, _ = A.shape
    a, b = A, B
    for d in range(int(math.ceil(math.log2(T)))):
        step = 2 ** d
        if step >= T:
            break
        # Positions >= step combine with the element `step` earlier
        # Monoid: (a2, b2) ∘ (a1, b1) = (a2*a1, a2*b1 + b2)
        b = torch.cat([b[:, :step, :],
                        a[:, step:, :] * b[:, :-step, :] + b[:, step:, :]], dim=1)
        a = torch.cat([a[:, :step, :],
                        a[:, step:, :] * a[:, :-step, :]], dim=1)
    return b


print("Utilities defined: spectral_sparsify (pairwise comparison, no topk),\n"
      "  spectral_to_spatial, spatial_to_spectral, spectral_proximal,\n"
      "  parallel_associative_scan (O(log T) Hillis-Steele)")

Utilities defined: spectral_sparsify (pairwise comparison, no topk),
  spectral_to_spatial, spatial_to_spectral, spectral_proximal,
  parallel_associative_scan (O(log T) Hillis-Steele)


In [4]:
# ── Spectral Token Embedding ─────────────────────────────────────────

class SpectralTokenEmbedding(nn.Module):
    """V12 embedding: tokens as sparse spectral configurations.
    
    The embedding table stores spectral magnitudes (real). Position enters
    as phase: each frequency bin gets phase offset omega_j * position.
    Per-subbundle top-s_k enforces spectral sparsity.
    
    Output: complex spectral section with exactly K*s_k active modes."""

    def __init__(self, cfg):
        super().__init__()
        self.cfg = cfg
        # Spectral magnitude embedding (real-valued)
        self.mag_embedding = nn.Embedding(cfg.vocab_size, cfg.fiber_dim)
        # Learned per-token phase offsets (real-valued, on top of positional phase)
        self.phase_embedding = nn.Embedding(cfg.vocab_size, cfg.fiber_dim)
        nn.init.uniform_(self.phase_embedding.weight, -math.pi, math.pi)
        
        # DFT frequencies for positional phase encoding
        # Per subbundle: omega_j = 2*pi*j/d_k for j = 0..d_k-1
        freqs = torch.zeros(cfg.fiber_dim)
        for k in range(cfg.n_subbundles):
            offset = k * cfg.subbundle_dim
            freqs[offset:offset+cfg.subbundle_dim] = (
                2 * math.pi * torch.arange(cfg.subbundle_dim).float() / cfg.subbundle_dim
            )
        self.register_buffer("freqs", freqs)

    def forward(self, token_ids):
        B, T = token_ids.shape
        cfg = self.cfg

        # Spectral magnitudes from embedding
        mag = self.mag_embedding(token_ids)           # (B, T, D) real
        phase_offset = self.phase_embedding(token_ids) # (B, T, D) real

        # Positional phase: omega_j * t
        positions = torch.arange(T, device=token_ids.device).float()
        pos_phase = positions.unsqueeze(-1) * self.freqs.unsqueeze(0)  # (T, D)

        # Complex spectral section: magnitude * exp(i * (learned_phase + positional_phase))
        total_phase = phase_offset + pos_phase.unsqueeze(0)  # (B, T, D)
        spectral = mag * torch.exp(1j * total_phase)  # (B, T, D) complex

        # Per-subbundle spectral sparsification
        spectral = spectral_sparsify(spectral, cfg)

        return spectral

In [5]:
# ── Context Accumulator (SSM) ────────────────────────────────────────

class ContextAccumulator(nn.Module):
    """Derives manifold coordinates q_t from the spatial projection of
    the spectral state. Uses a content-dependent SSM (parallel scan).
    
    q_t = A(x_t) * q_{t-1} + B(x_t) * psi(x_t)
    
    This is the Wilson line in disguise: content-dependent state
    accumulation that encodes the full path history."""

    def __init__(self, cfg):
        super().__init__()
        self.A_proj = nn.Linear(cfg.fiber_dim, cfg.context_dim)
        self.B_proj = nn.Linear(cfg.fiber_dim, cfg.context_dim)
        self.psi_proj = nn.Linear(cfg.fiber_dim, cfg.context_dim)

    def forward(self, x_spatial):
        """x_spatial: (B, T, fiber_dim) real spatial representation."""
        A = torch.sigmoid(self.A_proj(x_spatial))
        B = torch.sigmoid(self.B_proj(x_spatial))
        psi = self.psi_proj(x_spatial)
        return parallel_associative_scan(A, B * psi)

In [6]:
# ── Spectral Transport ───────────────────────────────────────────────

class SpectralTransport(nn.Module):
    """V12 spectral transport: direct multiply on the spectral section.
    
    X_tilde(omega) = X(omega) * exp(-D_k(ctx)*omega^2 - i*omega*A_k(ctx))
    
    Context-dependent diffusion D and gauge connection A warp the geometry
    frequency by frequency. The multiply is conceptually O(s) — only active
    modes matter — but implemented as element-wise on the full tensor
    (structural zeros remain zero)."""

    def __init__(self, cfg):
        super().__init__()
        self.cfg = cfg
        self.D_proj = nn.Linear(cfg.context_dim, cfg.fiber_dim)
        self.A_proj = nn.Linear(cfg.context_dim, cfg.fiber_dim)

    def forward(self, spectral_x, q_t):
        """spectral_x: (B, T, D) complex sparse spectral section.
        q_t: (B, T, context_dim) manifold coordinates."""
        cfg = self.cfg
        B, T, D = spectral_x.shape

        # Context-dependent transport coefficients
        diffusion = F.softplus(self.D_proj(q_t))  # (B, T, D) real, positive
        gauge = self.A_proj(q_t)                    # (B, T, D) real

        # Frequency grid per subbundle
        freqs = torch.fft.fftfreq(cfg.subbundle_dim, d=1.0, device=spectral_x.device)
        freqs = freqs.repeat(cfg.n_subbundles)  # (D,)
        w2 = freqs ** 2
        w = freqs

        # Transport kernel: exp(-D*w^2 - i*w*A)
        kernel = torch.exp(-diffusion * w2 - 1j * w * gauge)  # (B, T, D) complex

        return spectral_x * kernel

In [7]:
# ── Spectral Memory Bank ─────────────────────────────────────────────

class SpectralMemoryBank(nn.Module):
    """Memory atoms stored as spectral configurations (complex).
    Router selects atoms based on context + local spectral content.
    Selected atoms are IFFT'd to spatial for Hopfield comparison.
    
    Uses score-weighted atoms for fully differentiable routing — no topk.
    All atoms contribute, weighted by routing softmax scores.
    The Langevin Hopfield dynamics provide the second layer of selection."""

    def __init__(self, cfg):
        super().__init__()
        self.cfg = cfg
        sd = cfg.subbundle_dim
        K = cfg.n_subbundles
        A = cfg.atoms_per_subbundle

        # Spectral dictionary atoms stored as (real, imag) pairs
        self.dict_real = nn.ParameterList([
            nn.Parameter(torch.randn(A, sd) * 0.02) for _ in range(K)
        ])
        self.dict_imag = nn.ParameterList([
            nn.Parameter(torch.randn(A, sd) * 0.02) for _ in range(K)
        ])

        # Context projection for routing
        self.context_proj = nn.Sequential(
            nn.Linear(cfg.fiber_dim, cfg.context_dim), nn.SiLU()
        )
        # Per-subbundle routers (context + local spatial content)
        self.routers = nn.ModuleList([
            nn.Sequential(
                nn.Linear(cfg.context_dim + sd, A), nn.SiLU(), nn.Linear(A, A)
            ) for _ in range(K)
        ])

    def forward(self, q_t, dense_field):
        """Returns M_list: list of K tensors of score-weighted SPATIAL atom patterns.
        Each: (B*T, atoms_per_subbundle, subbundle_dim) real."""
        cfg = self.cfg
        sd = cfg.subbundle_dim
        B, T, _ = dense_field.shape
        BT = B * T

        ctx = self.context_proj(dense_field).reshape(BT, -1)
        field_chunks = dense_field.reshape(BT, cfg.n_subbundles, sd)

        # Pre-compute all spatial atoms in batch (eliminates K separate IFFTs)
        all_real = torch.stack(list(self.dict_real))   # (K, A, sd)
        all_imag = torch.stack(list(self.dict_imag))   # (K, A, sd)
        all_spatial = F.normalize(
            torch.fft.ifft(torch.complex(all_real, all_imag), dim=-1).real,
            dim=-1
        )  # (K, A, sd)

        M_list = []
        for k, router in enumerate(self.routers):
            # Route: context + local spatial content
            chunk_k = field_chunks[:, k, :]  # (BT, sd)
            logits = router(torch.cat([ctx, chunk_k], dim=-1))  # (BT, A)

            # Score-weighted atoms: fully differentiable, no topk needed
            # Routing softmax concentrates weight on relevant atoms;
            # Langevin Hopfield softmax provides second-layer selection
            scores = F.softmax(logits, dim=-1)  # (BT, A)
            M_k = all_spatial[k].unsqueeze(0) * scores.unsqueeze(-1)  # (BT, A, sd)
            M_list.append(M_k)

        return M_list

In [8]:
# ── Spectral Langevin Settler ────────────────────────────────────────

class SpectralLangevinSettler(nn.Module):
    """Reverse diffusion: dense spatial field → sparse spectral attractor.
    
    Settles in SPATIAL domain (Hopfield energy landscape is spatial geometry).
    Spectral proximal (FFT → top-s_k → IFFT) enforced at EVERY step.
    No noise during training (v11 pattern: noise only at eval for diversity)."""

    def __init__(self, cfg):
        super().__init__()
        self.cfg = cfg
        self.W_inh = nn.Parameter(torch.ones(cfg.fiber_dim) * 0.01)

    def forward(self, dense_field, M_list):
        """dense_field: (B, T, D) real dense spatial.
        M_list: list of K tensors, each (B*T, k_active, sd) real spatial atoms."""
        cfg = self.cfg
        B, T, D = dense_field.shape
        sd = cfg.subbundle_dim
        K = cfg.n_subbundles
        BT = B * T

        x = dense_field  # initialize from the cloud
        betas = torch.linspace(cfg.beta_init, cfg.beta_final, cfg.langevin_steps,
                               device=dense_field.device)

        # Stack all subbundle memories: (BT, K, k_active, sd)
        M_all = torch.stack(M_list, dim=1)

        for step in range(cfg.langevin_steps):
            beta = betas[step].item()

            # Vectorized Hopfield gradient across all K subbundles
            x_subs = x.reshape(BT, K, sd)
            sim = torch.einsum('bks,bkas->bka', x_subs, M_all)  # (BT, K, k_active)
            w = F.softmax(beta * sim, dim=-1)
            grad_E = -torch.einsum('bka,bkas->bks', w, M_all)  # (BT, K, sd)

            # Lateral inhibition (diagonal)
            inhib = self.W_inh * x

            # Langevin step
            x = x - cfg.langevin_lr * (grad_E.reshape(B, T, D) + inhib)

            # Annealing noise (eval only)
            if not self.training:
                x = x + math.sqrt(2.0 * cfg.langevin_lr / beta) * torch.randn_like(x)

            # SPECTRAL PROXIMAL at every step (the V12 key innovation)
            x = spectral_proximal(x, cfg)

        return x

In [9]:
# ── V12 Block ────────────────────────────────────────────────────────

class V12Block(nn.Module):
    """One complete forward-reverse spectral diffusion cycle.
    
    sparse spectral → spatial projection → context (SSM) →
    spectral transport → IFFT (dense field) → memory routing →
    Langevin settling (spatial, spectral proximal every step) →
    residual in spatial → FFT + re-sparsify → sparse spectral out"""

    def __init__(self, cfg):
        super().__init__()
        self.cfg = cfg
        self.context_acc = ContextAccumulator(cfg)
        self.transport = SpectralTransport(cfg)
        self.memory = SpectralMemoryBank(cfg)
        self.settler = SpectralLangevinSettler(cfg)
        self.norm = nn.LayerNorm(cfg.fiber_dim)
        # Residual gate: learned blend of settled output with input
        self.res_gate = nn.Parameter(torch.tensor(0.5))
        self.dropout = nn.Dropout(cfg.dropout)

    def forward(self, spectral_x):
        """spectral_x: (B, T, D) complex, spectrally sparse."""
        cfg = self.cfg

        # 1. Spatial projection for context accumulation
        x_spatial_in = spectral_to_spatial(spectral_x, cfg)

        # 2. Context accumulator (SSM on spatial)
        q_t = self.context_acc(x_spatial_in)

        # 3. Spectral transport (direct multiply on active modes)
        transported = self.transport(spectral_x, q_t)

        # 4. Field reconstruction: IFFT → dense spatial cloud
        dense_field = spectral_to_spatial(transported, cfg)
        dense_field = self.norm(dense_field)

        # 5. Memory routing from context + dense field
        M_list = self.memory(q_t, dense_field)

        # 6. Langevin settling (spatial domain, spectral proximal every step)
        settled = self.settler(dense_field, M_list)

        # 7. Gated residual in spatial domain, then FFT + re-sparsify
        gate = torch.sigmoid(self.res_gate)
        x_spatial_out = x_spatial_in + gate * self.dropout(settled)

        # Back to spectral and enforce sparsity
        spectral_out = spatial_to_spectral(x_spatial_out, cfg)
        spectral_out = spectral_sparsify(spectral_out, cfg)

        return spectral_out

In [10]:
# ── Full V12 Model ──────────────────────────────────────────────────

class V12Model(nn.Module):
    """V12: Spectral Sparsity CLM on Tiny Shakespeare."""

    def __init__(self, cfg):
        super().__init__()
        self.cfg = cfg
        self.embedding = SpectralTokenEmbedding(cfg)
        self.blocks = nn.ModuleList([V12Block(cfg) for _ in range(cfg.n_blocks)])
        self.final_norm = nn.LayerNorm(cfg.fiber_dim)
        self.decoder = nn.Sequential(
            nn.Linear(cfg.fiber_dim, cfg.fiber_dim), nn.SiLU(),
            nn.Dropout(cfg.dropout),
            nn.Linear(cfg.fiber_dim, cfg.vocab_size),
        )
        self.register_buffer(
            "block_loss_weights", torch.linspace(0.1, 1.0, cfg.n_blocks)
        )

    def _decode_spatial(self, spectral_x):
        """IFFT to spatial, norm, project to vocab logits."""
        spatial = spectral_to_spatial(spectral_x, self.cfg)
        return self.decoder(self.final_norm(spatial))

    def forward(self, token_ids):
        B, T = token_ids.shape
        cfg = self.cfg

        # 1. Spectral embedding
        spectral_x = self.embedding(token_ids)  # (B, T, D) complex, sparse

        # 2. Process through blocks with deep supervision
        intermediate_logits = []
        for block in self.blocks:
            spectral_x = block(spectral_x)
            logits = self._decode_spatial(spectral_x)[:, :-1, :]
            intermediate_logits.append(logits)

        # 3. Sparsity diagnostic
        spatial_final = spectral_to_spatial(spectral_x, cfg)
        spec_check = spatial_to_spectral(spatial_final, cfg)
        spec_sparsity = (spec_check.abs() < 1e-6).float().mean().item()

        info = {
            "spectral_sparsity": spec_sparsity,
            "intermediate_logits": intermediate_logits,
            "block_loss_weights": self.block_loss_weights,
        }
        return intermediate_logits[-1], info


# ── Instantiate and count parameters ─────────────────────────────────
model = V12Model(config).to(device)
n_params = sum(p.numel() for p in model.parameters())
n_embed = sum(p.numel() for p in model.embedding.parameters())
n_ctx = sum(sum(p.numel() for p in blk.context_acc.parameters()) for blk in model.blocks)
n_transport = sum(sum(p.numel() for p in blk.transport.parameters()) for blk in model.blocks)
n_memory = sum(sum(p.numel() for p in blk.memory.parameters()) for blk in model.blocks)
n_settler = sum(sum(p.numel() for p in blk.settler.parameters()) for blk in model.blocks)
n_decoder = sum(p.numel() for p in model.decoder.parameters()) + \
            sum(p.numel() for p in model.final_norm.parameters())
n_other = n_params - n_embed - n_ctx - n_transport - n_memory - n_settler - n_decoder

print(f"V12-Nano Total parameters: {n_params:,}")
print(f"  Spectral Embedding:        {n_embed:,} ({100*n_embed/n_params:.1f}%)")
print(f"  Context Accumulators ({config.n_blocks}):  {n_ctx:,} ({100*n_ctx/n_params:.1f}%)")
print(f"  Spectral Transport ({config.n_blocks}):    {n_transport:,} ({100*n_transport/n_params:.1f}%)")
print(f"  Memory Banks ({config.n_blocks}):          {n_memory:,} ({100*n_memory/n_params:.1f}%)")
print(f"  Langevin Settlers ({config.n_blocks}):     {n_settler:,} ({100*n_settler/n_params:.1f}%)")
print(f"  Decoder + norms:           {n_decoder:,} ({100*n_decoder/n_params:.1f}%)")
print(f"  Block norms/gates:         {n_other:,} ({100*n_other/n_params:.1f}%)")

V12-Nano Total parameters: 1,174,085
  Spectral Embedding:        33,280 (2.8%)
  Context Accumulators (4):  394,752 (33.6%)
  Spectral Transport (4):    264,192 (22.5%)
  Memory Banks (4):          395,776 (33.7%)
  Langevin Settlers (4):     1,024 (0.1%)
  Decoder + norms:           83,009 (7.1%)
  Block norms/gates:         2,052 (0.2%)


## Training

In [11]:
@torch.no_grad()
def estimate_loss(model, cfg, is_gpt=False):
    model.eval()
    results = {}
    for name, sd in [("train", train_data), ("val", val_data)]:
        tot_ce, tot_ok, tot_n, tot_sp = 0., 0, 0, 0.
        for _ in range(cfg.eval_steps):
            b = get_batch(sd, cfg)
            logits, info = model(b)
            tgt = b[:, 1:]
            ce = F.cross_entropy(logits.reshape(-1, cfg.vocab_size), tgt.reshape(-1))
            tot_ce += ce.item()
            tot_ok += (logits.argmax(-1) == tgt).sum().item()
            tot_n += tgt.numel()
            if not is_gpt:
                tot_sp += info.get("spectral_sparsity", 0.0)
        n = cfg.eval_steps
        results[name] = {
            "ce": tot_ce/n, "acc": tot_ok/tot_n,
            "sparsity": tot_sp/n if not is_gpt else 0.0,
        }
    model.train()
    return results


def train_model(model, cfg, label="V12", is_gpt=False):
    """Train loop for both V12 and GPT baseline."""
    optimizer = torch.optim.AdamW(model.parameters(), lr=cfg.learning_rate, weight_decay=0.05)

    def lr_lambda(step):
        if step < cfg.warmup_steps:
            return step / max(1, cfg.warmup_steps)
        progress = (step - cfg.warmup_steps) / max(1, cfg.max_steps - cfg.warmup_steps)
        return 0.5 * (1.0 + math.cos(math.pi * progress))

    scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)

    history = {
        "step": [], "train_ce": [], "val_ce": [],
        "train_acc": [], "val_acc": [],
        "train_bpc": [], "val_bpc": [],
        "sparsity": [], "lr": [],
        "step_times": [],
        "per_step_loss": [],
    }

    model.train()
    total_start = time.time()

    print(f"\nTraining {label}: {sum(p.numel() for p in model.parameters()):,} params")
    print(f"Steps: {cfg.max_steps}, Batch: {cfg.batch_size}, Seq: {cfg.seq_len}")
    print("=" * 70)

    pbar = tqdm(range(cfg.max_steps), desc=f"Training {label}")
    for step in pbar:
        # Eval
        if step % cfg.eval_interval == 0:
            res = estimate_loss(model, cfg, is_gpt=is_gpt)
            tr, vl = res["train"], res["val"]
            history["step"].append(step)
            history["train_ce"].append(tr["ce"])
            history["val_ce"].append(vl["ce"])
            history["train_acc"].append(tr["acc"])
            history["val_acc"].append(vl["acc"])
            history["train_bpc"].append(tr["ce"] / math.log(2))
            history["val_bpc"].append(vl["ce"] / math.log(2))
            history["sparsity"].append(vl["sparsity"])
            history["lr"].append(scheduler.get_last_lr()[0])
            sp_str = f" | Sp: {vl['sparsity']:.1%}" if not is_gpt else ""
            tqdm.write(f"[{label}] Step {step:5d} | Train CE: {tr['ce']:.3f} | "
                  f"Val CE: {vl['ce']:.3f} | Val BPC: {vl['ce']/math.log(2):.2f} | "
                  f"Val Acc: {vl['acc']:.1%}{sp_str}")

        # Train step
        step_start = time.time()
        batch = get_batch(train_data, cfg)
        optimizer.zero_grad()

        logits, info = model(batch)
        targets = batch[:, 1:]

        if is_gpt:
            loss = F.cross_entropy(logits.reshape(-1, cfg.vocab_size), targets.reshape(-1))
        else:
            # Deep supervision: weighted sum of per-block losses
            ce_loss = sum(
                w * F.cross_entropy(bl.reshape(-1, cfg.vocab_size), targets.reshape(-1))
                for bl, w in zip(info["intermediate_logits"], info["block_loss_weights"])
            ) / info["block_loss_weights"].sum()

            # Dictionary coherence regularization
            dcl, nd = 0., 0
            for blk in model.blocks:
                for dr, di in zip(blk.memory.dict_real, blk.memory.dict_imag):
                    # Coherence on spatial atoms
                    atoms = torch.fft.ifft(torch.complex(dr, di), dim=-1).real
                    An = F.normalize(atoms, dim=-1)
                    g = An @ An.T
                    dcl += (g - torch.eye(g.size(0), device=g.device)).pow(2).mean()
                    nd += 1
            dcl /= max(nd, 1)
            loss = ce_loss + 0.1 * dcl

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        scheduler.step()
        pbar.set_postfix(train_loss=f"{loss.item():.4f}")

        history["per_step_loss"].append(loss.item())
        history["step_times"].append(time.time() - step_start)

    total_time = time.time() - total_start

    # Final eval
    res = estimate_loss(model, cfg, is_gpt=is_gpt)
    tr, vl = res["train"], res["val"]
    history["step"].append(cfg.max_steps)
    history["train_ce"].append(tr["ce"])
    history["val_ce"].append(vl["ce"])
    history["train_acc"].append(tr["acc"])
    history["val_acc"].append(vl["acc"])
    history["train_bpc"].append(tr["ce"] / math.log(2))
    history["val_bpc"].append(vl["ce"] / math.log(2))
    history["sparsity"].append(vl["sparsity"])
    history["lr"].append(scheduler.get_last_lr()[0])

    print("=" * 70)
    print(f"[{label}] FINAL | Val CE: {vl['ce']:.3f} | Val BPC: {vl['ce']/math.log(2):.2f} | "
          f"Val Acc: {vl['acc']:.1%}")
    print(f"[{label}] Total time: {total_time:.1f}s | "
          f"Avg step: {np.mean(history['step_times'])*1000:.1f}ms")

    history["total_time"] = total_time
    history["avg_step_ms"] = np.mean(history["step_times"]) * 1000
    return history

## GPT-Nano Baseline

Minimal transformer for wall-clock comparison. Same data, batch size, sequence
length, and training steps as V12.

In [12]:
class GPTNano(nn.Module):
    """Minimal GPT for baseline comparison.
    Architecture: Embedding + [CausalAttention + MLP] × n_layer + LMHead"""

    def __init__(self, vocab_size=65, n_embd=128, n_head=4, n_layer=4, block_size=128, dropout=0.1):
        super().__init__()
        self.block_size = block_size
        self.tok_emb = nn.Embedding(vocab_size, n_embd)
        self.pos_emb = nn.Embedding(block_size, n_embd)
        self.drop = nn.Dropout(dropout)

        self.blocks = nn.ModuleList()
        for _ in range(n_layer):
            self.blocks.append(nn.ModuleDict({
                'ln1': nn.LayerNorm(n_embd),
                'attn_qkv': nn.Linear(n_embd, 3 * n_embd),
                'attn_proj': nn.Linear(n_embd, n_embd),
                'ln2': nn.LayerNorm(n_embd),
                'mlp_fc1': nn.Linear(n_embd, 4 * n_embd),
                'mlp_fc2': nn.Linear(4 * n_embd, n_embd),
            }))

        self.ln_f = nn.LayerNorm(n_embd)
        self.lm_head = nn.Linear(n_embd, vocab_size, bias=False)
        self.n_head = n_head
        self.n_embd = n_embd

        causal_mask = torch.tril(torch.ones(block_size, block_size))
        self.register_buffer('causal_mask', causal_mask.view(1, 1, block_size, block_size))

    def forward(self, idx):
        B, T = idx.shape
        pos = torch.arange(T, device=idx.device)
        x = self.drop(self.tok_emb(idx) + self.pos_emb(pos))

        head_dim = self.n_embd // self.n_head
        for block in self.blocks:
            # Attention
            h = block['ln1'](x)
            qkv = block['attn_qkv'](h).reshape(B, T, 3, self.n_head, head_dim)
            q, k, v = qkv.unbind(2)  # each (B, T, n_head, head_dim)
            q, k, v = q.transpose(1, 2), k.transpose(1, 2), v.transpose(1, 2)
            att = (q @ k.transpose(-2, -1)) * (head_dim ** -0.5)
            att = att.masked_fill(self.causal_mask[:, :, :T, :T] == 0, float('-inf'))
            att = F.softmax(att, dim=-1)
            y = (att @ v).transpose(1, 2).reshape(B, T, self.n_embd)
            x = x + block['attn_proj'](y)
            # MLP
            h = block['ln2'](x)
            x = x + block['mlp_fc2'](F.gelu(block['mlp_fc1'](h)))

        logits = self.lm_head(self.ln_f(x))
        return logits[:, :-1, :], {}


gpt_model = GPTNano(
    vocab_size=vocab_size, n_embd=128, n_head=4, n_layer=4,
    block_size=config.seq_len, dropout=config.dropout
).to(device)

gpt_params = sum(p.numel() for p in gpt_model.parameters())
print(f"GPT-Nano parameters: {gpt_params:,}")
print(f"V12-Nano parameters: {n_params:,}")
print(f"Ratio: {gpt_params/n_params:.1f}×")

GPT-Nano parameters: 826,368
V12-Nano parameters: 1,174,085
Ratio: 0.7×


In [13]:
# ── Train V12 ──
v12_history = train_model(model, config, label="V12", is_gpt=False)


Training V12: 1,174,085 params
Steps: 5000, Batch: 32, Seq: 128


Training V12:   0%|          | 0/5000 [00:00<?, ?it/s]

[V12] Step     0 | Train CE: 4.192 | Val CE: 4.194 | Val BPC: 6.05 | Val Acc: 1.3% | Sp: 38.7%
[V12] Step   250 | Train CE: 2.824 | Val CE: 2.827 | Val BPC: 4.08 | Val Acc: 24.0% | Sp: 39.3%
[V12] Step   500 | Train CE: 2.312 | Val CE: 2.330 | Val BPC: 3.36 | Val Acc: 33.5% | Sp: 44.2%
[V12] Step   750 | Train CE: 2.088 | Val CE: 2.124 | Val BPC: 3.06 | Val Acc: 38.1% | Sp: 48.3%
[V12] Step  1000 | Train CE: 1.979 | Val CE: 2.035 | Val BPC: 2.94 | Val Acc: 40.1% | Sp: 49.0%
[V12] Step  1250 | Train CE: 1.882 | Val CE: 1.976 | Val BPC: 2.85 | Val Acc: 41.8% | Sp: 49.3%
[V12] Step  1500 | Train CE: 1.812 | Val CE: 1.929 | Val BPC: 2.78 | Val Acc: 43.3% | Sp: 49.2%
[V12] Step  1750 | Train CE: 1.748 | Val CE: 1.889 | Val BPC: 2.72 | Val Acc: 44.0% | Sp: 48.7%
[V12] Step  2000 | Train CE: 1.719 | Val CE: 1.874 | Val BPC: 2.70 | Val Acc: 44.4% | Sp: 48.6%
[V12] Step  2250 | Train CE: 1.693 | Val CE: 1.837 | Val BPC: 2.65 | Val Acc: 45.2% | Sp: 48.4%
[V12] Step  2500 | Train CE: 1.677 | Val 

KeyboardInterrupt: 

In [ ]:
# ── Train GPT-Nano ──
gpt_model = GPTNano(
    vocab_size=vocab_size, n_embd=128, n_head=4, n_layer=4,
    block_size=config.seq_len, dropout=config.dropout
).to(device)

gpt_history = train_model(gpt_model, config, label="GPT", is_gpt=True)


Training GPT: 1,072,128 params
Steps: 5000, Batch: 32, Seq: 2048


Training GPT:   0%|          | 0/5000 [00:00<?, ?it/s]

[GPT] Step     0 | Train CE: 4.385 | Val CE: 4.387 | Val BPC: 6.33 | Val Acc: 1.4%


## Results Comparison

In [ ]:
# ── Side-by-side comparison ─────────────────────────────────────────

fig, axes = plt.subplots(3, 3, figsize=(18, 15))

# 1. Val CE vs step
axes[0,0].plot(v12_history["step"], v12_history["val_ce"], 'b-o', ms=3, label='V12')
axes[0,0].plot(gpt_history["step"], gpt_history["val_ce"], 'r-s', ms=3, label='GPT-Nano')
axes[0,0].set_xlabel('Step'); axes[0,0].set_ylabel('Val CE')
axes[0,0].set_title('Validation Cross-Entropy vs Step')
axes[0,0].legend(); axes[0,0].grid(True, alpha=0.3)

# 2. Val BPC vs step
axes[0,1].plot(v12_history["step"], v12_history["val_bpc"], 'b-o', ms=3, label='V12')
axes[0,1].plot(gpt_history["step"], gpt_history["val_bpc"], 'r-s', ms=3, label='GPT-Nano')
axes[0,1].set_xlabel('Step'); axes[0,1].set_ylabel('Val BPC')
axes[0,1].set_title('Bits per Character vs Step')
axes[0,1].legend(); axes[0,1].grid(True, alpha=0.3)

# 3. Val Acc vs step
axes[0,2].plot(v12_history["step"], v12_history["val_acc"], 'b-o', ms=3, label='V12')
axes[0,2].plot(gpt_history["step"], gpt_history["val_acc"], 'r-s', ms=3, label='GPT-Nano')
axes[0,2].set_xlabel('Step'); axes[0,2].set_ylabel('Val Accuracy')
axes[0,2].set_title('Validation Accuracy vs Step')
axes[0,2].legend(); axes[0,2].grid(True, alpha=0.3)

# 4. Val BPC vs wall-clock time
v12_cumtime = np.cumsum([0] + [v12_history["avg_step_ms"]/1000 * config.eval_interval] * (len(v12_history["step"])-1))
gpt_cumtime = np.cumsum([0] + [gpt_history["avg_step_ms"]/1000 * config.eval_interval] * (len(gpt_history["step"])-1))
axes[1,0].plot(v12_cumtime, v12_history["val_bpc"], 'b-o', ms=3, label='V12')
axes[1,0].plot(gpt_cumtime, gpt_history["val_bpc"], 'r-s', ms=3, label='GPT-Nano')
axes[1,0].set_xlabel('Wall-clock time (s)'); axes[1,0].set_ylabel('Val BPC')
axes[1,0].set_title('BPC vs Wall-Clock Time')
axes[1,0].legend(); axes[1,0].grid(True, alpha=0.3)

# 5. Step time distribution
axes[1,1].hist(np.array(v12_history["step_times"])*1000, bins=50, alpha=0.7, label='V12', color='blue')
axes[1,1].hist(np.array(gpt_history["step_times"])*1000, bins=50, alpha=0.7, label='GPT-Nano', color='red')
axes[1,1].set_xlabel('Step time (ms)'); axes[1,1].set_ylabel('Count')
axes[1,1].set_title('Step Time Distribution')
axes[1,1].legend(); axes[1,1].grid(True, alpha=0.3)


# 6. Per-Step Train Loss
window = 100
axes[1,2].plot(v12_history.get("per_step_loss", []), 'b-', alpha=0.3, label='V12')
if len(v12_history.get("per_step_loss", [])) >= window:
    v12_smooth = np.convolve(v12_history["per_step_loss"], np.ones(window)/window, mode='valid')
    axes[1,2].plot(np.arange(window-1, len(v12_history["per_step_loss"])), v12_smooth, 'b-', lw=2, label='V12 (Smooth)')
axes[1,2].plot(gpt_history.get("per_step_loss", []), 'r-', alpha=0.3, label='GPT-Nano')
if len(gpt_history.get("per_step_loss", [])) >= window:
    gpt_smooth = np.convolve(gpt_history["per_step_loss"], np.ones(window)/window, mode='valid')
    axes[1,2].plot(np.arange(window-1, len(gpt_history["per_step_loss"])), gpt_smooth, 'r-', lw=2, label='GPT-Nano (Smooth)')
axes[1,2].set_xlabel('Step'); axes[1,2].set_ylabel('Train Loss')
axes[1,2].set_title('Per-Step Train Loss')
axes[1,2].legend(); axes[1,2].grid(True, alpha=0.3)

axes[2,0].axis('off')
axes[2,2].axis('off')

# 7. Summary table as text
axes[2,1].axis('off')
summary = (
    f"{'Metric':<25} {'V12':>10} {'GPT':>10}\n"
    f"{'─'*45}\n"
    f"{'Parameters':<25} {n_params:>10,} {gpt_params:>10,}\n"
    f"{'Avg step (ms)':<25} {v12_history['avg_step_ms']:>10.1f} {gpt_history['avg_step_ms']:>10.1f}\n"
    f"{'Total time (s)':<25} {v12_history['total_time']:>10.1f} {gpt_history['total_time']:>10.1f}\n"
    f"{'Final Val CE':<25} {v12_history['val_ce'][-1]:>10.3f} {gpt_history['val_ce'][-1]:>10.3f}\n"
    f"{'Final Val BPC':<25} {v12_history['val_bpc'][-1]:>10.2f} {gpt_history['val_bpc'][-1]:>10.2f}\n"
    f"{'Final Val Acc':<25} {v12_history['val_acc'][-1]:>9.1%} {gpt_history['val_acc'][-1]:>9.1%}\n"
    f"{'Speed ratio':<25} {gpt_history['avg_step_ms']/v12_history['avg_step_ms']:>9.1f}×\n"
)
axes[2,1].text(0.05, 0.95, summary, transform=axes[2,1].transAxes,
               fontsize=11, verticalalignment='top', fontfamily='monospace',
               bbox=dict(boxstyle='round', facecolor='lightyellow', alpha=0.8))
axes[2,1].set_title('Summary')

plt.suptitle('V12 Spectral Sparsity vs GPT-Nano — Training Comparison',
             fontsize=15, fontweight='bold')
plt.tight_layout(); plt.show()

## Diagnostics

In [ ]:
# ── V12 Diagnostics ─────────────────────────────────────────────────

@torch.no_grad()
def v12_diagnostics(model, cfg):
    model.eval()
    fig, axes = plt.subplots(2, 3, figsize=(18, 10))

    # --- 1. Spectral sparsity verification ---
    test_batch = get_batch(val_data, cfg)
    spectral_emb = model.embedding(test_batch)
    emb_sparsity = (spectral_emb.abs() < 1e-6).float().mean().item()

    block_sparsities = []
    x = spectral_emb
    for blk in model.blocks:
        x = blk(x)
        block_sparsities.append((x.abs() < 1e-6).float().mean().item())

    labels = ['Embedding'] + [f'Block {i}' for i in range(cfg.n_blocks)]
    sparsities = [emb_sparsity] + block_sparsities
    expected = 1 - cfg.spectral_sparsity / cfg.subbundle_dim

    axes[0,0].bar(labels, sparsities, color='steelblue', edgecolor='black')
    axes[0,0].axhline(y=expected, color='red', linestyle='--',
                      label=f'Expected: {expected:.0%}')
    axes[0,0].set_ylabel('Fraction of zeros (spectral)')
    axes[0,0].set_title('Spectral Sparsity Through Network')
    axes[0,0].set_ylim(0, 1); axes[0,0].legend(); axes[0,0].grid(True, alpha=0.3, axis='y')

    # --- 2. Langevin trajectory: spectral sparsity at each step ---
    spectral_emb = model.embedding(test_batch)
    x_spatial_in = spectral_to_spatial(spectral_emb, cfg)
    q_t = model.blocks[0].context_acc(x_spatial_in)
    transported = model.blocks[0].transport(spectral_emb, q_t)
    dense_field = model.blocks[0].norm(spectral_to_spatial(transported, cfg))
    M_list = model.blocks[0].memory(q_t, dense_field)

    B, T, D = dense_field.shape
    sd, K = cfg.subbundle_dim, cfg.n_subbundles
    BT = B * T
    x_lang = dense_field.clone()
    betas = torch.linspace(cfg.beta_init, cfg.beta_final, cfg.langevin_steps,
                           device=device)
    M_all = torch.stack(M_list, dim=1)

    step_sp = []
    field_sp = (spatial_to_spectral(x_lang, cfg).abs() < 1e-6).float().mean().item()
    step_sp.append(field_sp)  # before settling

    for s in range(cfg.langevin_steps):
        beta = betas[s].item()
        x_subs = x_lang.reshape(BT, K, sd)
        sim = torch.einsum('bks,bkas->bka', x_subs, M_all)
        w = F.softmax(beta * sim, dim=-1)
        grad_E = -torch.einsum('bka,bkas->bks', w, M_all)
        x_lang = x_lang - cfg.langevin_lr * grad_E.reshape(B, T, D)
        x_lang = spectral_proximal(x_lang, cfg)
        sp = (spatial_to_spectral(x_lang, cfg).abs() < 1e-6).float().mean().item()
        step_sp.append(sp)

    axes[0,1].plot(range(len(step_sp)), step_sp, 'b-o', lw=2, ms=6)
    axes[0,1].axhline(y=expected, color='red', linestyle='--', label=f'Target: {expected:.0%}')
    axes[0,1].set_xlabel('Langevin Step (0=init from field)')
    axes[0,1].set_ylabel('Spectral sparsity (frac zeros)')
    axes[0,1].set_title('Spectral Sparsity Through Langevin')
    axes[0,1].legend(); axes[0,1].grid(True, alpha=0.3)

    # --- 3. Spectral fingerprint heatmap for selected tokens ---
    chars_to_show = list('abcdefghij AROM:\n')
    char_ids = [stoi.get(c, 0) for c in chars_to_show]
    ids_tensor = torch.tensor(char_ids, device=device).unsqueeze(0)  # (1, N)
    with torch.no_grad():
        spec = model.embedding.mag_embedding(ids_tensor)[0]  # (N, D)
    # Reshape to subbundles and show first 2
    spec_subs = spec.reshape(len(char_ids), cfg.n_subbundles, cfg.subbundle_dim)
    axes[0,2].imshow(spec_subs[:, 0, :].abs().cpu().numpy(), aspect='auto', cmap='viridis')
    axes[0,2].set_yticks(range(len(chars_to_show)))
    axes[0,2].set_yticklabels([repr(c) for c in chars_to_show], fontsize=8)
    axes[0,2].set_xlabel('Frequency bin (subbundle 0)')
    axes[0,2].set_title('Spectral Magnitudes (Subbundle 0)')

    # --- 4. Context sensitivity test ---
    text_a = "The brave knight drew his sword and charged into battle"
    text_b = "The quiet child drew his picture and smiled at mother"
    T_test = min(cfg.seq_len, len(text_a), len(text_b))
    ids_a = torch.tensor(encode(text_a[:T_test]), dtype=torch.long).unsqueeze(0).to(device)
    ids_b = torch.tensor(encode(text_b[:T_test]), dtype=torch.long).unsqueeze(0).to(device)

    spec_a = model.embedding(ids_a)
    spec_b = model.embedding(ids_b)
    spa_a = spectral_to_spatial(spec_a, cfg)
    spa_b = spectral_to_spatial(spec_b, cfg)
    q_a = model.blocks[0].context_acc(spa_a)
    q_b = model.blocks[0].context_acc(spa_b)
    ctx_div = (q_a - q_b).norm(dim=-1)[0].cpu().numpy()

    axes[1,0].plot(ctx_div, 'b-', lw=2)
    axes[1,0].set_xlabel('Position'); axes[1,0].set_ylabel('||q_a - q_b||')
    axes[1,0].set_title('Context Divergence\n(different histories → different manifold coords)')
    axes[1,0].grid(True, alpha=0.3)

    # --- 5. Spectral overlap matrix (token similarity via shared modes) ---
    sample_chars = list('abcdefghijklmnop')
    sample_ids = torch.tensor([stoi.get(c, 0) for c in sample_chars], device=device).unsqueeze(0)
    spec_vecs = model.embedding(sample_ids)[0]  # (N, D) complex
    # Overlap: |sum_w x*(w) y(w)| for each pair
    N = len(sample_chars)
    overlap = torch.zeros(N, N)
    for i in range(N):
        for j in range(N):
            overlap[i, j] = (spec_vecs[i].conj() * spec_vecs[j]).abs().sum().item()
    overlap = overlap / overlap.diag().sqrt().unsqueeze(1) / overlap.diag().sqrt().unsqueeze(0)

    axes[1,1].imshow(overlap.cpu().numpy(), cmap='hot', vmin=0, vmax=1)
    axes[1,1].set_xticks(range(N)); axes[1,1].set_xticklabels(sample_chars)
    axes[1,1].set_yticks(range(N)); axes[1,1].set_yticklabels(sample_chars)
    axes[1,1].set_title('Spectral Overlap Matrix\n(token similarity via shared frequencies)')

    # --- 6. Spatial waveforms of token embeddings ---
    wave_chars = list('aeiou ')
    wave_ids = torch.tensor([stoi.get(c, 0) for c in wave_chars], device=device).unsqueeze(0)
    wave_spec = model.embedding(wave_ids)[0]  # (N, D) complex
    for i, c in enumerate(wave_chars):
        spatial_wave = spectral_to_spatial(wave_spec[i:i+1].unsqueeze(0), cfg)[0, 0]
        axes[1,2].plot(spatial_wave[:32].cpu().numpy(), label=repr(c), alpha=0.8)
    axes[1,2].set_xlabel('Spatial dimension (first 32)')
    axes[1,2].set_ylabel('Amplitude')
    axes[1,2].set_title('Spatial Waveforms of Token Embeddings\n(IFFT of sparse spectral sections)')
    axes[1,2].legend(fontsize=8); axes[1,2].grid(True, alpha=0.3)

    plt.suptitle('V12 Diagnostics: Spectral Sparsity + Geometry',
                 fontsize=14, fontweight='bold')
    plt.tight_layout(); plt.show()

    print(f"Embedding spectral sparsity: {emb_sparsity:.1%}")
    print(f"Block output sparsities: {[f'{s:.1%}' for s in block_sparsities]}")
    print(f"Langevin sparsity trajectory: {[f'{s:.1%}' for s in step_sp]}")

v12_diagnostics(model, config)

In [ ]:
# ── Text Generation ─────────────────────────────────────────────────

@torch.no_grad()
def generate_text(model, prompt_str, cfg, max_new=200, temperature=0.8):
    model.eval()
    ids = torch.tensor(encode(prompt_str), dtype=torch.long).unsqueeze(0).to(device)
    for _ in range(max_new):
        ctx = ids[:, -cfg.seq_len:] if ids.size(1) >= cfg.seq_len else ids
        logits, _ = model(ctx)
        probs = F.softmax(logits[:, -1, :] / temperature, dim=-1)
        next_id = torch.multinomial(probs, 1)
        ids = torch.cat([ids, next_id], dim=1)
    return decode(ids[0].tolist())


print("=" * 60)
print("V12 TEXT GENERATION (temperature=0.8)")
print("=" * 60)
for p in ["ROMEO:\n", "To be or not to ", "The king ", "O, "]:
    print(f"\n--- Prompt: {repr(p)} ---")
    print(generate_text(model, p, config))

print("\n" + "=" * 60)
print("GPT-NANO TEXT GENERATION (temperature=0.8)")
print("=" * 60)
for p in ["ROMEO:\n", "To be or not to ", "The king ", "O, "]:
    print(f"\n--- Prompt: {repr(p)} ---")
    print(generate_text(gpt_model, p, config))

## Summary

### V12: Spectral Sparsity on the Cotangent Bundle

**Core change from V11:** Sparsity moved from spatial fiber to spectral (Fourier) fiber.

**What this means:**
- Tokens are sparse collections of frequency modes, not sparse spatial activations
- Transport acts directly on active modes: O(s) instead of O(d log d)
- Field reconstruction (diffusion) IS the IFFT from spectral to spatial
- The proximal operator enforces spectral sparsity (FFT → top-s_k → IFFT)
- Routing via spectral overlap — no pairwise attention

**V12 Non-Negotiables:**
1. Sparse in spectral, dense only transiently in spatial
2. Field reconstruction IS the inverse Fourier transform
3. Langevin starts from the reconstructed (IFFT) field
4. Context warps the spectral metric (D and A are context-dependent)
5. Spectral proximal at every Langevin step
6. Subbundles are independent spectral channels
7. No pairwise attention